### Install Libraries

In [ ]:
# %pip install ddgs trafilatura
%pip install openai-agents

### Setup

In [ ]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

MODEL="gpt-4.1-mini"

### Step 1: Define the Tools

In [ ]:
@function_tool # Instructs OpenAI SDK to set these up as tools for agents
def search_web(query: str):
    """Search the web using DuckDuckGo Browser. Returns 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2)

In [ ]:
#'beautiful soup' is another package you can use if you need more control
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(f"  \n \u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
        return text
    else:
        print(f"  \u274C Failed to fetch or extract text from {url}")
        return f"Could not extract text from {url}. Try a different source."

In [ ]:
from openai import OpenAI
openai_client = OpenAI()

import base64
import uuid
from pathlib import Path

from agents import function_tool

IMAGE_DIR = Path("generated_images")
IMAGE_DIR.mkdir(exist_ok=True)

@function_tool
def generate_image(prompt: str) -> str:
    """Generate a hero image from a detailed visual description.
    Returns the file path of the saved image."""
    try:
        response = openai_client.images.generate(
            prompt=prompt,
            model="gpt-image-2.5-flare",
            quality="medium",
        )
        img_bytes = base64.b64decode(response.data[0].b64_json)

        path = IMAGE_DIR / f"{uuid.uuid4().hex}.png"
        path.write_bytes(img_bytes)

        return f"Image saved to {path}"
    except Exception as e:
        return f"Error generating image: {e}"

### Step 2: The Agents

The Agent Prompts tell the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce



#### Research Agent

In [ ]:

RESEARCH_AGENT_PROMPT="""You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You MUST gather information from at least 3 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching. ALLOW yourself moments to simply think on what has already been gathered to determine the best next step.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

*** IMPORTANT:
After each search with search_web, you MUST first explain your reasoning:
- Which URLs seem most relevant and why?
- Which ones you will fetch and why?
- Which ones you will discard and why?
Only after writing out your reasoning should you call fetch_url.
***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs. If a URL doesn't provide useful information, discard it but note why.
4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url]
)

research_agent_as_tool = research_agent.as_tool(
        tool_name="research_agent",
        tool_description="Research a topic and return a brief with key facts, statistic themes, and source URLs. Pass topic as input",
        max_turns=20 # max turns are not inherited by the orchestrator agent and defaults to 10
    )

#### Image Generator Agent

In [ ]:
IMAGE_AGENT_PROMPT = """You are an image prompt specialist. Given a topic and content summary,
craft a detailed prompt for a hero image.

Rules for your prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No human faces or recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.

IMPORTANT: When returning the image URL, copy it EXACTLY character by character. Do not modify, shorten, or paraphase the URL.
"""

image_agent = Agent(
    name="Image Agent",
    instructions=IMAGE_AGENT_PROMPT,
    model=MODEL,
    tools=[generate_image]
)

image_agent_as_tool = image_agent.as_tool(
        tool_name="image_agent",
        tool_description="Generate a hero image for an article based on a supplied topic and content summary.",
        max_turns=20 # max turns are not inherited by the orchestrator agent and defaults to 10
    )

#### Orchestrator Agent

In [ ]:
# My original prompt
# ORCHESTRATOR_AGENT_PROMPT = """
# You are an orchestrator for multi agent researchers. Your job is to take in various research briefs and select the best one for the publication of an article.

# You MUST call for at least two completed research briefs. 

# You have access to a tool:
# - research_agent: Create a research brief based on a query

# Your typical process:
# 1. Pass your query to your research agent and obtain a research brief
# 2. Reflect on the research results — which sources look most relevant and why?
# 3. Call your research agent again for a second brief
# 4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
# 5. If you feel the research isn't enough or too similar, call your research agent again to create additional briefs.
# 6. When you have enough information, decide on the best research brief to return.

# Good research brief MUST include:
# - Key facts and statistics
# - Main themes and arguments from the sources
# - Notable data points
# - Source URLs for attribution

# Until you are ready, just keep working — reflecting as much as you need to.
# Do not rush.
# """
ORCHESTRATOR_AGENT_PROMPT = """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.


Once you have selected the best brief, you MUST use the image_agent to generate a SINGLE image. 
Use the research brief to supply the image agent with the topic and content summary it needs to generate the image.

Deliver the selected research brief as the final output, and include the image at th top of your final output in markdown format like this: ![hero image](generated_images/image.png)S

IMPORTANT: When returning the image path, copy it EXACTLY character by character. Do not modify, shorten, or paraphase the URL.
"""

orchestrator_agent = Agent(
    name="Ochestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini", ## reasoning model
    tools=[research_agent_as_tool, image_agent_as_tool]
)

### Let's Run it!

In [ ]:
from agents import trace

with trace("Article Writer", group_id="Learning AI Engineering", metadata={"userID": "user123"}):
    result = await Runner.run(
        orchestrator_agent,
        input = "Research the following topic and produce a comprehensive research brief: How will AI be used in Healthcare in 2030-2035",
        max_turns=30
    )

In [ ]:
print(f"Agent: {result.last_agent.name}")
print(f"---")
display(Markdown(result.final_output))

### Trace logs

https://platform.openai.com/home -> logs -> traces -> Agent workflow